# Baseline comparison — my system vs MicroRCA, RCACopilot, RCAgent

Faithful re-implementations of three RCA baselines evaluated on a live edge1 cluster.
Two benches:
- **20 real labeled K8s incidents** (dataset/, gpt-5 judge) — broad fault spectrum.
- **Mesh bench** (Online Boutique + Istio, 6 injected latency/CPU faults) — MicroRCA's home turf.

Systems: `yours_raw` (collect-all), `yours_distilled` (Evidence-First pipeline),
`RCACopilot` (retrieval few-shot), `RCAgent` (self-consistency over N tool-augmented samples),
`MicroRCA` (attributed graph + personalized PageRank; degraded resource-only on the 20).

**Headline:** on the real 20, my system leads (raw 90%, distilled 85% at ~1/3 the tokens);
RCAgent ties distilled at 85% but costs ~10x tokens and ~3x latency; RCACopilot 75%.
MicroRCA is metric-only and covers almost none of the real fault spectrum (5% top-1).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- summarized results (baked in; no data files needed) ----
# 20 real incidents (gpt-5 judge, n=20 each)
real20 = {
    'yours_raw':       {'acc': 90.0, 'prec': 90.0, 'tok': 11580, 'lat': 39.5},
    'yours_distilled': {'acc': 85.0, 'prec': 85.0, 'tok':  3631, 'lat': 37.9},
    'RCACopilot':      {'acc': 75.0, 'prec': 75.0, 'tok': 12186, 'lat': 28.0},
    'RCAgent':         {'acc': 85.0, 'prec': 85.0, 'tok': 34742, 'lat': 111.8},
}
microrca20 = {'top1': 5.0, 'top3': 40.0, 'structural_blind': 7}  # degraded, resource-only

# mesh bench (6 latency/CPU faults on Online Boutique)
mesh = {
    'yours_raw':       {'acc': 100.0, 'tok': 11530, 'lat': 43.0},
    'yours_distilled': {'acc':  50.0, 'tok':  3328, 'lat': 47.2},
    'RCACopilot':      {'acc': 100.0, 'tok': 12247, 'lat': 31.6},
    'RCAgent':         {'acc': 100.0, 'tok': 34588, 'lat': 127.6},
}
mesh_microrca = {'top1': 33.3, 'top3': 83.3}  # faithful MicroRCA

# accuracy by fault category on the 20 (%)
cats = ['Admission', 'Scheduling', 'Startup', 'Runtime', 'Service Routing', 'Performance']
by_cat = {
    'yours_raw':       [100, 100,  66.7, 100,  66.7, 100],
    'yours_distilled': [100, 100,  66.7,  75, 100,    50],
    'RCACopilot':      [100,  75,  66.7,  75, 100,     0],
    'RCAgent':         [100, 100,  66.7,  75,  66.7, 100],
}

SYS = ['yours_raw', 'yours_distilled', 'RCACopilot', 'RCAgent']
COL = {'yours_raw': '#f0871e', 'yours_distilled': '#2f7dc4',
       'RCACopilot': '#5aa469', 'RCAgent': '#a05fc0', 'MicroRCA': '#c0504d'}

def lbl(ax, bars, fmt='%.0f'):
    ax.bar_label(bars, fmt=fmt, fontsize=8, padding=2)

In [ ]:
# ===== Figure 1: 20 real incidents — accuracy + cost =====
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

# panel 0: localization accuracy (LLM systems + MicroRCA top-1/top-3)
labels = SYS + ['MicroRCA\n(top-1)', 'MicroRCA\n(top-3)']
vals   = [real20[s]['acc'] for s in SYS] + [microrca20['top1'], microrca20['top3']]
colors = [COL[s] for s in SYS] + [COL['MicroRCA'], '#d98682']
b = ax[0].bar(labels, vals, color=colors)
lbl(ax[0], b, '%.0f%%')
ax[0].set_title('Localization accuracy — 20 real K8s incidents'); ax[0].set_ylim(0, 108)
ax[0].set_ylabel('% incidents root-caused (gpt-5 judge)')
ax[0].axhline(90, ls='--', lw=.8, color='#f0871e', alpha=.5)
ax[0].tick_params(axis='x', labelsize=8); ax[0].grid(alpha=.3, axis='y')

# panel 1: prompt tokens (log)
b = ax[1].bar(SYS, [real20[s]['tok'] for s in SYS], color=[COL[s] for s in SYS])
lbl(ax[1], b, '%.0f')
ax[1].set_yscale('log'); ax[1].set_ylim(1000, 60000)
ax[1].set_title('Prompt context (mean tokens)'); ax[1].set_ylabel('tokens (log)')
ax[1].tick_params(axis='x', labelsize=8, rotation=15); ax[1].grid(alpha=.3, axis='y', which='both')

# panel 2: latency
b = ax[2].bar(SYS, [real20[s]['lat'] for s in SYS], color=[COL[s] for s in SYS])
lbl(ax[2], b, '%.0fs')
ax[2].set_title('Diagnosis latency (mean s)'); ax[2].set_ylabel('seconds')
ax[2].set_ylim(0, 130); ax[2].tick_params(axis='x', labelsize=8, rotation=15); ax[2].grid(alpha=.3, axis='y')

plt.suptitle('20 real incidents:  my system (raw 90%, distilled 85% @ ~1/3 tokens)  vs  '
             'RCAgent (85% @ 10x tokens)  vs  RCACopilot (75%)  vs  MicroRCA (5% top-1)', fontsize=11)
plt.tight_layout(); plt.savefig('cmp_real20.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ===== Figure 2: mesh bench (MicroRCA home turf) + accuracy by category =====
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))

# panel 0: mesh bench accuracy
labels = SYS + ['MicroRCA\n(top-1)', 'MicroRCA\n(top-3)']
vals   = [mesh[s]['acc'] for s in SYS] + [mesh_microrca['top1'], mesh_microrca['top3']]
colors = [COL[s] for s in SYS] + [COL['MicroRCA'], '#d98682']
b = ax[0].bar(labels, vals, color=colors)
lbl(ax[0], b, '%.0f%%')
ax[0].set_title('Mesh bench — 6 latency/CPU faults (MicroRCA\'s home turf)'); ax[0].set_ylim(0, 108)
ax[0].set_ylabel('% culprit service localized')
ax[0].tick_params(axis='x', labelsize=8); ax[0].grid(alpha=.3, axis='y')
ax[0].annotate('pipeline over-compresses\nsmall telemetry', xy=(1, 50), xytext=(1.1, 20),
               fontsize=8, color='#2f7dc4', ha='left',
               arrowprops=dict(arrowstyle='->', color='#2f7dc4', lw=.8))

# panel 1: accuracy by fault category (grouped)
x = np.arange(len(cats)); w = 0.2
for i, s in enumerate(SYS):
    ax[1].bar(x + (i-1.5)*w, by_cat[s], w, color=COL[s], label=s)
ax[1].set_xticks(x); ax[1].set_xticklabels(cats, fontsize=8, rotation=20, ha='right')
ax[1].set_title('Accuracy by fault category — 20 real incidents'); ax[1].set_ylim(0, 112)
ax[1].set_ylabel('% root-caused'); ax[1].legend(fontsize=8, ncol=2); ax[1].grid(alpha=.3, axis='y')

plt.suptitle('Mesh bench: LLM systems 100% (distilled 50%) vs MicroRCA 33%/83%.  '
             'By category: Performance splits the field; RCACopilot=0% (no similar past incident).', fontsize=10)
plt.tight_layout(); plt.savefig('cmp_mesh_category.png', dpi=150, bbox_inches='tight'); plt.show()